In [1]:
# 03_targeting_frontier.ipynb
# Module 3: fractional-knapsack targeting order under cost structures, and the
# efficiency-equity frontier (reads results/tables/m2_sensitivity.csv).
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, pyreadstat
import seaborn as sns, matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
# --- grayscale publication style ---
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"axes.edgecolor":"0.15","grid.color":"0.85","font.size":10,
                     "axes.titlesize":1,"figure.autolayout":True})
GRAY = ["0.20","0.45","0.65","0.80"]
# --- resolve folders relative to repo (no path is printed, for anonymity) ---
BASE = Path("..") if (Path("../data")).exists() else Path(".")
DATA, FIG, TAB = BASE/"data", BASE/"results/figures", BASE/"results/tables"
FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
def savefig(fig, name):
    # save both formats at 600 dpi, no caption/title; paths are never printed
    for ext in ("png","pdf"): fig.savefig(FIG/(name+"."+ext), dpi=600, bbox_inches="tight")
    plt.close(fig)
PVS=[f"PV{i}CIL" for i in range(1,6)]
def wR2(y,yh,w):
    yb=np.average(y,weights=w); return 1-np.sum(w*(y-yh)**2)/np.sum(w*(y-yb)**2)

R=pd.read_csv(TAB/"m2_sensitivity.csv"); R["grp"]=R.SES+"_"+R.SEX; tot=R.wN.sum()
def costs(kind):
    if kind=="equal": return np.ones(len(R))
    if kind=="gap":   return (1+(R.base_CIL.max()-R.base_CIL)/(R.base_CIL.max()-R.base_CIL.min())).values
    if kind=="size":  return (R.wN/R.wN.mean()).values
orders={}
for k in ["equal","gap","size"]:
    order=np.argsort(-(R.dCIL.values/costs(k))); orders[k]=list(R.grp.values[order])
pd.DataFrame({k:orders[k] for k in orders}).to_csv(TAB/"m3_targeting_order.csv",index=False)
print("Targeting order (equal cost):", " > ".join(orders["equal"]))
# efficiency-equity frontier across budget (fraction of population reached)
def simulate(order_idx,phi):
    budget=phi*tot; cov=np.zeros(len(R)); rem=budget
    for i in order_idx:
        take=min(R.wN.values[i],rem); cov[i]=take/R.wN.values[i]; rem-=take
        if rem<=0: break
    newCIL=R.base_CIL.values+cov*R.dCIL.values
    gain=np.sum(cov*R.wN.values*R.dCIL.values)/tot
    low=R.SES.values=="Low"; high=R.SES.values=="High"
    gap=np.average(newCIL[high],weights=R.wN.values[high])-np.average(newCIL[low],weights=R.wN.values[low])
    return gain,gap
eff=np.argsort(-R.dCIL.values); eq=np.argsort(R.base_CIL.values)
out=[]
for phi in [0.1,0.2,0.33,0.5,0.75,1.0]:
    ge,gape=simulate(eff,phi); gq,gapq=simulate(eq,phi)
    out.append(dict(budget=phi,eff_gain=ge,eff_gap=gape,eq_gain=gq,eq_gap=gapq,
                    eff_loss_pct=100*(gq-ge)/ge if ge>0 else 0))
F=pd.DataFrame(out); F.to_csv(TAB/"m3_frontier.csv",index=False)
print(F.round(3).to_string(index=False))
# figure: grayscale diminishing-returns / efficiency~equity curves (no title/caption)
fig,ax=plt.subplots(figsize=(4.4,3.3))
ax.plot(F.budget,F.eff_gain,marker="o",color="0.15",lw=1.8,label="Efficiency-first")
ax.plot(F.budget,F.eq_gain,marker="s",color="0.55",lw=1.5,ls="--",label="Equity-first")
ax.set_xlabel("Budget (fraction of population reached)"); ax.set_ylabel("Mean CIL gain")
ax.legend(frameon=False,fontsize=8)
savefig(fig,"fig_m3_frontier")


Targeting order (equal cost): Low_Sex0 > Low_Sex1 > Mid_Sex0 > High_Sex0 > Mid_Sex1 > High_Sex1
 budget  eff_gain  eff_gap  eq_gain  eq_gap  eff_loss_pct
   0.10     2.619   33.140    2.619  33.140         0.000
   0.20     5.150   25.575    5.087  27.784        -1.220
   0.33     8.190   16.485    7.869  27.784        -3.923
   0.50    11.838   16.169   11.835  16.266        -0.023
   0.75    16.771   26.772   16.771  26.772         0.000
   1.00    20.884   34.206   20.884  34.206         0.000
